In [53]:
import fitz
import re

def extract_clean_text(pdf_path):
    doc = fitz.open(pdf_path)
    full_text = ""

    for page_num in range(len(doc)):
        page = doc[page_num]
        text = page.get_text()


        text = re.sub(r'WORK AND ENERGY', '', text)


        text = re.sub(r'\n\d+\n', '\n', text)


        text = re.sub(r'\s+', ' ', text)

        full_text += text + "\n"

    return full_text.strip()



raw_text = extract_clean_text("data/iesc110.pdf")

print(raw_text[:1500])

In the previous few chapters we have talked about ways of describing the motion of objects, the cause of motion and gravitation. Another concept that helps us understand and interpret many natural phenomena is ‘work’. Closely related to work are energy and power. In this chapter we shall study these concepts. All living beings need food. Living beings have to perform several basic activities to survive. We call such activities ‘life processes’. The energy for these processes comes from food. We need energy for other activities like playing, singing, reading, writing, thinking, jumping, cycling and running. Activities that are strenuous require more energy. Animals too get engaged in activities. For example, they may jump and run. They have to fight, move away from enemies, find food or find a safe place to live. Also, we engage some animals to lift weights, carry loads, pull carts or plough fields. All such activities require energy. Think of machines. List the machines that you have c

In [56]:
def clean_text(text):
    import re

    text = text.replace('\r', '\n')

    # Fix common OCR mojibake before regex cleanup.
    text = text.replace('â€¢', '\u2022').replace('â€˜', "'").replace('â€™', "'")

    # Remove recurring header/footer fragments.
    text = re.sub(r'(?i)\bWORK\s+AND\s+ENERGY\b', ' ', text)
    text = re.sub(r'(?i)\bWORK\s+ENERGY\b', ' ', text)
    text = re.sub(r'(?i)\bReprint\s+\d{4}\s*[-–]\s*\d{2}\s+SCIENCE\b', ' ', text)
    text = re.sub(r'(?i)\bhapter\b|\bchapter\b', ' ', text)
    text = re.sub(r'(?i)\b(?:ORK|NERGY|E|C|AND)\b(?:\s+\b(?:ORK|NERGY|E|C|AND)\b){3,}', ' ', text)

    # Remove standalone page numbers.
    text = re.sub(r'(?m)^\s*\d{1,3}\s*$', ' ', text)

    # Put each bullet on its own line for stable block splitting.
    text = re.sub(r'\s*\u2022\s*', '\n\u2022 ', text)

    # Remove immediate duplicate words from OCR noise.
    text = re.sub(r'\b(\w+)\s+\1\b', r'\1', text, flags=re.IGNORECASE)

    # Normalize spacing while preserving line breaks.
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{2,}', '\n', text)

    return text.strip()

In [57]:
def structure_text(text):
    # Add explicit separators around structural markers.
    text = re.sub(r'(?i)\bExample\s+\d+(?:\.\d+)+\b', '\n[EXAMPLE]\n', text)
    text = re.sub(r'(?i)\bActivity\s*_+\s*\d+(?:\.\d+)*\b', '\n[ACTIVITY]\n', text)
    text = re.sub(r'(?i)\bQuestions\b\s*:?', '\n[QUESTION]\n', text)

    # Mark numbered section headings only when they look like headings.
    text = re.sub(r'\b10\.\d+(?:\.\d+)?\b', '\n[SECTION]\n', text)

    # Final spacing cleanup after marker insertion.
    text = re.sub(r'\n{2,}', '\n', text)
    return text.strip()


structured_text = structure_text(cleaned_text)

print(structured_text[:1500])

In the previous few chapters we have talked about ways of describing the motion of objects, the cause of motion gravitation. Another concept that helps us understand interpret many natural phenomena is ‘work’. Closely related to work are energy power. In this we shall study these concepts. All living beings need food. Living beings have to perform several basic activities to survive. We call such activities ‘life processes’. The energy for these processes comes from food. We need energy for other activities like playing, singing, reading, writing, thinking, jumping, cycling running. Activities that are strenuous require more energy. Animals too get engaged in activities. For example, they may jump run. They have to fight, move away from enemies, find food or find a safe place to live. Also, we engage some animals to lift weights, carry loads, pull carts or plough fields. All such activities require energy. Think of machines. List the machines that you have come across. What do they nee

In [59]:
def structure_text(text):
    # Add explicit separators around structural markers.
    text = re.sub(r'(?i)\bExample\s+\d+(?:\.\d+)+\b', '\n[EXAMPLE]\n', text)
    text = re.sub(r'(?i)\bActivity\s*_+\s*\d+(?:\.\d+)*\b', '\n[ACTIVITY]\n', text)
    text = re.sub(r'(?i)\bQuestions\b\s*:?', '\n[QUESTION]\n', text)

    # Mark numbered section headings only when they look like headings.
    text = re.sub(r'\b10\.\d+(?:\.\d+)?\b', '\n[SECTION]\n', text)

    # Final spacing cleanup after marker insertion.
    text = re.sub(r'\n{2,}', '\n', text)
    return text.strip()

In [60]:
def split_into_blocks(text):
    blocks = []
    current_block = ""
    current_type = "CONCEPT"

    marker_to_type = {
        "[SECTION]": "SECTION",
        "[EXAMPLE]": "EXAMPLE",
        "[ACTIVITY]": "ACTIVITY",
        "[QUESTION]": "QUESTION",
    }

    for raw_line in text.split('\n'):
        line = raw_line.strip()
        if not line:
            continue

        if line in marker_to_type:
            if current_block:
                blocks.append((current_type, current_block.strip()))
                current_block = ""
            current_type = marker_to_type[line]
            continue

        # Keep each bullet as a separate ACTIVITY block.
        if current_type == "ACTIVITY" and line.startswith("\u2022") and current_block:
            blocks.append((current_type, current_block.strip()))
            current_block = line
            continue

        if current_block:
            current_block += " " + line
        else:
            current_block = line

    if current_block:
        blocks.append((current_type, current_block.strip()))

    return blocks

In [61]:
cleaned_text = clean_text(raw_text)
structured_text = structure_text(cleaned_text)
blocks = split_into_blocks(structured_text)

In [62]:
for b in blocks[:5]:
    print("\nTYPE:", b[0])
    print("TEXT:", b[1][:200])


TYPE: CONCEPT
TEXT: In the previous few chapters we have talked about ways of describing the motion of objects, the cause of motion and gravitation. Another concept that helps us understand and interpret many natural phe

TYPE: SECTION
TEXT: Work What is work? There is a difference in the way we use the term ‘work’ in day-to-day life and the way we use it in science. To make this point clear let us consider a few examples.

TYPE: SECTION
TEXT: NOT MUCH ‘WORK’ IN SPITE OF WORKING HARD! Kamali is preparing for examinations. She spends lot of time in studies. She reads books, draws diagrams, organises her thoughts, collects question papers, at

TYPE: ACTIVITY
TEXT: • We have discussed in the above paragraphs a number of activities which we normally consider to be work ENERGY

TYPE: ACTIVITY
TEXT: • Think of situations when the object is not displaced in spite of a force acting on it.


In [67]:
sample_texts = [chunk["text"][:200] for chunk in chunks[:5]]

In [68]:
from transformers import GPT2Tokenizer, BertTokenizer, T5Tokenizer

gpt2_tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
bert_tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
t5_tokenizer = T5Tokenizer.from_pretrained("t5-small")

C:\Users\abcom\PycharmProjects\PythonProject\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\abcom\PycharmProjects\PythonProject\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\abcom\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to a

In [69]:
for i, text in enumerate(sample_texts):
    print(f"\n--- SAMPLE {i+1} ---")
    print("TEXT:", text[:100])

    gpt2_tokens = gpt2_tokenizer.tokenize(text)
    bert_tokens = bert_tokenizer.tokenize(text)
    t5_tokens = t5_tokenizer.tokenize(text)

    print("\nGPT-2 Tokens:", len(gpt2_tokens))
    print("BERT Tokens:", len(bert_tokens))
    print("T5 Tokens:", len(t5_tokens))

    print("\nExample Tokens:")
    print("GPT-2:", gpt2_tokens[:10])
    print("BERT:", bert_tokens[:10])
    print("T5:", t5_tokens[:10])


--- SAMPLE 1 ---
TEXT: In the previous few chapters we have talked about ways of describing the motion of objects, the caus

GPT-2 Tokens: 37
BERT Tokens: 38
T5 Tokens: 39

Example Tokens:
GPT-2: ['In', 'Ġthe', 'Ġprevious', 'Ġfew', 'Ġchapters', 'Ġwe', 'Ġhave', 'Ġtalked', 'Ġabout', 'Ġways']
BERT: ['in', 'the', 'previous', 'few', 'chapters', 'we', 'have', 'talked', 'about', 'ways']
T5: ['▁In', '▁the', '▁previous', '▁few', '▁chapters', '▁we', '▁have', '▁talked', '▁about', '▁ways']

--- SAMPLE 2 ---
TEXT: • We have discussed in the above paragraphs a number of activities which we normally consider to be 

GPT-2 Tokens: 22
BERT Tokens: 21
T5 Tokens: 25

Example Tokens:
GPT-2: ['âĢ¢', 'ĠWe', 'Ġhave', 'Ġdiscussed', 'Ġin', 'Ġthe', 'Ġabove', 'Ġparagraphs', 'Ġa', 'Ġnumber']
BERT: ['•', 'we', 'have', 'discussed', 'in', 'the', 'above', 'paragraph', '##s', 'a']
T5: ['▁•', '▁We', '▁have', '▁discussed', '▁in', '▁the', '▁above', '▁paragraph', 's', '▁']

--- SAMPLE 3 ---
TEXT: • Think of situations wh